# `thoi` — Scalable O-information via Gaussian Copula

> **Docs**: https://pypi.org/project/thoi

**`thoi`** computes higher-order information measures by applying a **Gaussian copula approximation**.
This makes it orders of magnitude faster than KSG-based approaches, allowing it to sweep **all multiplets** of a large dataset in a single call.

### Key idea
Any multivariate distribution is first mapped to a standard Gaussian via the empirical CDF (copula normalisation), then the Gaussian analytic formulas for TC, DTC, and $\Omega$ are applied. This is exact for Gaussian data and a fast approximation otherwise.

### What we compute
| Measure | Column in output | Notes |
|---------|-----------------|-------|
| Total Correlation | `tc` | |
| Dual Total Correlation | `dtc` | |
| O-information | `o` | $\Omega = TC - DTC$ |
| S-information | `s` | $S = TC + DTC$ |

The output is a **DataFrame** — one row per multiplet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import random
from sklearn.datasets import load_wine
from sklearn.feature_selection import mutual_info_classif

from thoi.measures.gaussian_copula import multi_order_measures

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print("Imports OK.")


## 1. Artificial Data — Sanity Check

Same controlled triplets as the other notebooks.


In [ ]:
n_features, n_samples = 3, 500
eta = 0.1

source = random.normal(0, 1, n_samples)
data_red = np.array([source,
                     source + random.normal(0, 1, n_samples) * eta,
                     source + random.normal(0, 1, n_samples) * eta])

s0 = random.normal(0, 1, n_samples)
s1 = random.normal(0, 1, n_samples)
data_syn = np.array([s0 + random.normal(0, 1, n_samples) * eta,
                     s1 + random.normal(0, 1, n_samples) * eta,
                     s0 + s1 + random.normal(0, 1, n_samples) * eta])

X_red = data_red.T   # (500, 3)  — thoi expects (n_samples, n_features)
X_syn = data_syn.T
print("Input shapes:", X_red.shape, X_syn.shape)


## 2. Compute O-information with `thoi`

`multi_order_measures` returns a DataFrame with one row per multiplet.


In [ ]:
df_red = multi_order_measures(X_red, min_order=3, max_order=3)
df_syn = multi_order_measures(X_syn, min_order=3, max_order=3)

print("Output DataFrame (redundant):")
print(df_red.to_string(index=False))
print()

omega_thoi_red = float(df_red['o'].iloc[0])
omega_thoi_syn = float(df_syn['o'].iloc[0])
print(f"O-information  Ω(X0, X1, X2)")
print(f"  Redundant:    {omega_thoi_red:+.4f}   ← POSITIVE ✓")
print(f"  Synergistic:  {omega_thoi_syn:+.4f}   ← NEGATIVE ✓")


## 3. Information Landscape — Scaling to More Variables

The real advantage of `thoi` is sweeping **all multiplets** across multiple orders in a single call.
We build a 10-variable redundant system and map every multiplet of size 2–5.


In [ ]:
n_big = 10
source_big = random.normal(0, 1, n_samples)
X_big = np.column_stack([source_big + random.normal(0, 1, n_samples) * eta
                         for _ in range(n_big)])   # (500, 10)

df_big = multi_order_measures(X_big, min_order=2, max_order=5)
print(f"Total multiplets evaluated: {len(df_big)}")
print(df_big.groupby('order')['o'].describe().round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
palette = sns.color_palette('viridis', n_colors=4)
for k, col in zip(range(2, 6), palette):
    subset = df_big[df_big['order'] == k].reset_index(drop=True)
    if subset.empty:
        continue
    ax.scatter(range(len(subset)), subset['o'],
               s=18, alpha=0.7, color=col, label=f'order {k}')
ax.axhline(0, color='black', lw=0.9, ls='--', label='$\\Omega=0$')
ax.set_xlabel('Multiplet index (within each order)')
ax.set_ylabel('O-information $\\Omega$')
ax.set_title('thoi: information landscape — 10-variable redundant system\n'
             '(positive = redundant, negative = synergistic)', fontweight='bold')
ax.legend(title='Multiplet size', fontsize=9)
plt.tight_layout()
plt.show()


## 4. Real Data — UCI Wine Dataset

We apply `thoi` to all multiplets of size 3–5 from the top-5 features.


In [ ]:
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feature_names = list(wine.feature_names)

mi_scores = mutual_info_classif(X_wine, y_wine, random_state=SEED)
order = np.argsort(mi_scores)[::-1]
top5 = [feature_names[i] for i in order[:5]]
X_top5 = X_wine[:, order[:5]]
print("Top-5 features:", top5)

df_wine = multi_order_measures(X_top5, min_order=3, max_order=5)
oinfo_arr = df_wine['o'].values
print(f"\nMultiplets: {len(df_wine)}")
print(f"Ω range: [{oinfo_arr.min():.4f}, {oinfo_arr.max():.4f}]")
print(f"Mean Ω: {oinfo_arr.mean():.4f}  ({'redundancy-dominated' if oinfo_arr.mean() > 0 else 'synergy-dominated'})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution histogram
axes[0].hist(oinfo_arr, bins=20, color=sns.color_palette()[0],
             edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='black', lw=1.5, ls='--', label=r'$\Omega=0$')
axes[0].axvline(oinfo_arr.mean(), color='tomato', lw=1.5, ls='-',
                label=f'Mean={oinfo_arr.mean():.3f}')
axes[0].set_xlabel(r'O-information $\Omega$')
axes[0].set_ylabel('Count')
axes[0].set_title('O-information distribution\n(Wine top-5 features)', fontweight='bold')
axes[0].legend()

# Mean Ω by order
mean_by_order = df_wine.groupby('order')['o'].mean()
axes[1].bar(mean_by_order.index, mean_by_order.values,
            color=sns.color_palette()[:len(mean_by_order)])
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_xlabel('Multiplet order')
axes[1].set_ylabel('Mean $\\Omega$')
axes[1].set_title('Mean O-information by order', fontweight='bold')
axes[1].set_xticks(mean_by_order.index)

plt.tight_layout()
plt.show()
